# Next-token loss, made observable

A randomly initialized nano GPT-2, built with Hugging Face `GPT2LMHeadModel` and the GPT-2 tokenizer. The trunk is two layers wide and 64-dimensional, so the whole notebook runs on a CPU. The vocabulary is the real GPT-2 vocabulary (50,257), which is the number an untrained model has to land near.

The harness we start from is:

```python
hidden = model(tokens)
logits = output_head(hidden)
loss = cross_entropy(
    logits[:, :-1].reshape(-1, vocab_size),
    tokens[:, 1:].reshape(-1),
)
```

That shift is the right idea. It is not yet a correct training loss: padding still votes, a packed document boundary still votes, and `model(tokens)` on `GPT2LMHeadModel` is already logits rather than hidden states. Each section below fixes one of those and prints the evidence, including the token strings, so an off-by-one cannot hide in a row of ids.


In [1]:
try:
    import transformers
    from transformers.loss.loss_utils import ForCausalLMLoss
except Exception:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.48"])
    import transformers
    from transformers.loss.loss_utils import ForCausalLMLoss

import inspect
import math
import weakref

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils._python_dispatch import TorchDispatchMode
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

print("transformers", transformers.__version__)
print("torch", torch.__version__)

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
# GPT-2 ships with no pad id. The eos string is a real end-of-document token, so it
# cannot also be detected as padding by its id. Padding is whatever attention_mask marks.
print("pad_token is", tokenizer.pad_token)
print("eos_token is", repr(tokenizer.eos_token), "id", tokenizer.eos_token_id)
print("padding_side is", repr(tokenizer.padding_side))
tokenizer.pad_token = tokenizer.eos_token

config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=256,
    n_embd=64,
    n_layer=2,
    n_head=4,
    n_inner=256,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    bos_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    tie_word_embeddings=True,
)
# transformers routes GPT2LMHeadModel.loss through ForCausalLMLoss when this is set.
# Left as None, forward still falls back to that function, but it warns on every call.
config.loss_type = "ForCausalLM"

model = GPT2LMHeadModel(config).to(device)
model.eval()
vocab_size = model.config.vocab_size
print("vocab_size", vocab_size)
print("lm_head", model.lm_head)
print("tied storage", model.lm_head.weight.data_ptr() == model.transformer.wte.weight.data_ptr())


transformers 5.19.0
torch 2.7.0
device cpu
pad_token is None
eos_token is '<|endoftext|>' id 50256
padding_side is 'right'
vocab_size 50257
lm_head Linear(in_features=64, out_features=50257, bias=False)
tied storage True


## Shapes

`GPT2LMHeadModel.forward` already applies `lm_head` and returns logits. The hidden states are the transformer's `last_hidden_state`, after the final layer norm. The head is a bias-free linear map from `n_embd` to `vocab_size`.


In [2]:
def show(name, tensor, meanings):
    print(f"{name}: shape={tuple(tensor.shape)}  dtype={tensor.dtype}")
    if tensor.ndim == 0:
        print("  scalar: one number, the mean negative log-likelihood in nats")
        return
    if len(meanings) != tensor.ndim:
        raise AssertionError((name, tuple(tensor.shape), meanings))
    for axis, meaning in enumerate(meanings):
        print(f"  dim {axis} = {tensor.shape[axis]}: {meaning}")


def token_string(token_id):
    token_id = int(token_id)
    if token_id == -100:
        return "<ignored>"
    return repr(tokenizer.decode([token_id]))


def pad_batch(sequences, pad_id):
    width = max(len(seq) for seq in sequences)
    ids = torch.full((len(sequences), width), pad_id, dtype=torch.long, device=device)
    mask = torch.zeros(len(sequences), width, dtype=torch.long, device=device)
    for row, seq in enumerate(sequences):
        ids[row, : len(seq)] = torch.tensor(seq, dtype=torch.long, device=device)
        mask[row, : len(seq)] = 1
    return ids, mask


def shift_loss(logits, labels):
    # labels use -100 for any target that must not vote.
    # logits[:, :-1] lines up with labels[:, 1:]: position t scores token t+1.
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    flat_logits = shift_logits.reshape(-1, vocab_size).float()
    flat_labels = shift_labels.reshape(-1)
    loss = F.cross_entropy(flat_logits, flat_labels, ignore_index=-100)
    return loss, shift_logits, shift_labels, flat_logits, flat_labels


sentence = "The capital of France is Paris."
input_ids = tokenizer(sentence, return_tensors="pt").input_ids.to(device)
attention_mask = torch.ones_like(input_ids)

with torch.no_grad():
    hidden = model.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
    logits = model.lm_head(hidden)
    labels = input_ids.clone()
    loss, shift_logits, shift_labels, flat_logits, flat_labels = shift_loss(logits, labels)

show("input_ids", input_ids, ["batch", "sequence positions, each one a token id"])
show("attention_mask", attention_mask, ["batch", "sequence positions, 1 if the token is real"])
show("hidden", hidden, ["batch", "sequence positions", "model dimension n_embd"])
show("logits", logits, ["batch", "sequence positions", "vocabulary scores, one per token id"])
show("shift_logits", shift_logits, [
    "batch",
    "positions 0 through T-2; position t is about to be scored against token t+1",
    "vocabulary scores",
])
show("shift_labels", shift_labels, ["batch", "tokens 1 through T-1, the target at each scored position"])
show("flat_logits", flat_logits, ["one row per scored position in the batch", "vocabulary scores"])
show("flat_labels", flat_labels, ["one target id per scored position"])
show("loss", loss, [])
print("loss value", round(loss.item(), 4))


input_ids: shape=(1, 7)  dtype=torch.int64
  dim 0 = 1: batch
  dim 1 = 7: sequence positions, each one a token id
attention_mask: shape=(1, 7)  dtype=torch.int64
  dim 0 = 1: batch
  dim 1 = 7: sequence positions, 1 if the token is real
hidden: shape=(1, 7, 64)  dtype=torch.float32
  dim 0 = 1: batch
  dim 1 = 7: sequence positions
  dim 2 = 64: model dimension n_embd
logits: shape=(1, 7, 50257)  dtype=torch.float32
  dim 0 = 1: batch
  dim 1 = 7: sequence positions
  dim 2 = 50257: vocabulary scores, one per token id
shift_logits: shape=(1, 6, 50257)  dtype=torch.float32
  dim 0 = 1: batch
  dim 1 = 6: positions 0 through T-2; position t is about to be scored against token t+1
  dim 2 = 50257: vocabulary scores
shift_labels: shape=(1, 6)  dtype=torch.int64
  dim 0 = 1: batch
  dim 1 = 6: tokens 1 through T-1, the target at each scored position
flat_logits: shape=(6, 50257)  dtype=torch.float32
  dim 0 = 6: one row per scored position in the batch
  dim 1 = 50257: vocabulary scores
fl

## The shift, in token strings

Position `t` is fed token `t` and scored against token `t+1`. The last position has no following token, so it is dropped rather than trained. Reading the strings is the check: the target column is the input column moved up by one word, not a wall of ids.


In [3]:
ids_row = input_ids[0]
targets_row = shift_labels[0]
print(f"{'pos':>4}  {'input token':<24}  {'target token':<24}")
print("-" * 56)
for t in range(ids_row.shape[0]):
    inp = token_string(ids_row[t])
    if t < targets_row.shape[0]:
        tgt = token_string(targets_row[t])
    else:
        tgt = "<dropped: no next token>"
    print(f"{t:4d}  {inp:<24}  {tgt:<24}")

# The target at position t must be the input at position t+1, as text, not merely as an id.
for t in range(targets_row.shape[0]):
    assert token_string(targets_row[t]) == token_string(ids_row[t + 1])
print("shift check: every target string is the next input string")


 pos  input token               target token            
--------------------------------------------------------
   0  'The'                     ' capital'              
   1  ' capital'                ' of'                   
   2  ' of'                     ' France'               
   3  ' France'                 ' is'                   
   4  ' is'                     ' Paris'                
   5  ' Paris'                  '.'                     
   6  '.'                       <dropped: no next token>
shift check: every target string is the next input string


## Padding

Two documents of different lengths share one batch width. GPT-2's pad id is the end-of-text id, so a pad cannot be detected by looking at the integer. `attention_mask` is 0 on the padded positions. Those positions are set to `-100` before the shift, and `cross_entropy(..., ignore_index=-100)` drops them from the mean. The count of contributing targets has to change, or the mask is not actually applied.


In [4]:
short = "Hello."
long = "The capital of France is Paris."
pad_id = tokenizer.pad_token_id
rows = [
    tokenizer.encode(short, add_special_tokens=False),
    tokenizer.encode(long, add_special_tokens=False),
]
pad_ids, pad_mask = pad_batch(rows, pad_id)
print("pad id", pad_id, "equals eos id", tokenizer.eos_token_id, ":", pad_id == tokenizer.eos_token_id)

with torch.no_grad():
    pad_hidden = model.transformer(input_ids=pad_ids, attention_mask=pad_mask).last_hidden_state
    pad_logits = model.lm_head(pad_hidden)

show("pad_ids", pad_ids, ["batch (two documents)", "sequence positions, right-padded to the longer document"])
show("pad_mask", pad_mask, ["batch", "1 on real tokens, 0 on padding"])
show("pad_hidden", pad_hidden, ["batch", "sequence positions", "model dimension"])
show("pad_logits", pad_logits, ["batch", "sequence positions", "vocabulary scores"])

unmasked_labels = pad_ids.clone()
_, _, unmasked_targets, unmasked_flat_logits, unmasked_flat_targets = shift_loss(pad_logits, unmasked_labels)
contributing_before = int(unmasked_flat_targets.numel())

masked_labels = pad_ids.clone()
masked_labels[pad_mask == 0] = -100
pad_loss, _, masked_targets, _, masked_flat_targets = shift_loss(pad_logits, masked_labels)
contributing_after = int((masked_flat_targets != -100).sum().item())

show("unmasked_targets", unmasked_targets, ["batch", "shifted targets, padding ids still included"])
show("masked_targets", masked_targets, ["batch", "shifted targets, padding set to -100"])
print("contributing tokens before the padding mask:", contributing_before)
print("contributing tokens after the padding mask: ", contributing_after)
assert contributing_after < contributing_before

for row in range(pad_ids.shape[0]):
    print()
    print(f"batch row {row}")
    print(f"{'pos':>4}  {'input token':<24}  {'target token':<24}  counted")
    print("-" * 68)
    for t in range(pad_ids.shape[1] - 1):
        counted = "yes" if int(masked_targets[row, t]) != -100 else "no"
        print(
            f"{t:4d}  {token_string(pad_ids[row, t]):<24}  "
            f"{token_string(masked_targets[row, t]):<24}  {counted}"
        )
    print(
        f"{pad_ids.shape[1] - 1:4d}  {token_string(pad_ids[row, -1]):<24}  "
        f"{'<dropped: no next token>':<24}  no"
    )


pad id 50256 equals eos id 50256 : True
pad_ids: shape=(2, 7)  dtype=torch.int64
  dim 0 = 2: batch (two documents)
  dim 1 = 7: sequence positions, right-padded to the longer document
pad_mask: shape=(2, 7)  dtype=torch.int64
  dim 0 = 2: batch
  dim 1 = 7: 1 on real tokens, 0 on padding
pad_hidden: shape=(2, 7, 64)  dtype=torch.float32
  dim 0 = 2: batch
  dim 1 = 7: sequence positions
  dim 2 = 64: model dimension
pad_logits: shape=(2, 7, 50257)  dtype=torch.float32
  dim 0 = 2: batch
  dim 1 = 7: sequence positions
  dim 2 = 50257: vocabulary scores
unmasked_targets: shape=(2, 6)  dtype=torch.int64
  dim 0 = 2: batch
  dim 1 = 6: shifted targets, padding ids still included
masked_targets: shape=(2, 6)  dtype=torch.int64
  dim 0 = 2: batch
  dim 1 = 6: shifted targets, padding set to -100
contributing tokens before the padding mask: 12
contributing tokens after the padding mask:  7

batch row 0
 pos  input token               target token              counted
-----------------------

## Perplexity of the untrained model

Cross-entropy here is a natural log. Perplexity is `exp(loss)`. A model that puts equal mass on every vocabulary item has loss `log(V)` and perplexity `V`. This model has not been trained. If the number is nowhere near the vocabulary size, the shift, the reduction, or an accidental softmax is wrong, and nothing below is worth reading.


In [5]:
perplexity_text = (
    "Elephants are the largest living land animals. Three living species are recognised. "
    "They have a trunk, tusks, and large ears. The trunk brings food and water to the mouth. "
    "African elephants have larger ears than Asian elephants. They live in savannahs, forests, "
    "and marshes, and they stay near water when they can."
)
perp_ids = tokenizer(perplexity_text, return_tensors="pt").input_ids.to(device)
assert perp_ids.shape[1] <= config.n_positions, perp_ids.shape
perp_labels = perp_ids.clone()
with torch.no_grad():
    perp_hidden = model.transformer(input_ids=perp_ids).last_hidden_state
    perp_logits = model.lm_head(perp_hidden)
    perp_loss, _, _, _, _ = shift_loss(perp_logits, perp_labels)
untrained_perplexity = float(perp_loss.exp())
uniform_loss = math.log(vocab_size)
print("scored tokens", perp_ids.shape[1] - 1)
print("loss nats", round(perp_loss.item(), 4))
print("log(vocab_size)", round(uniform_loss, 4))
print("perplexity", round(untrained_perplexity, 2))
print("vocab_size", vocab_size)
ratio_to_vocab = untrained_perplexity / vocab_size
print("perplexity / vocab_size", round(ratio_to_vocab, 3))
assert 0.5 < ratio_to_vocab < 2.0, (untrained_perplexity, vocab_size)
print("untrained perplexity sits near the vocabulary size")


scored tokens 69
loss nats 10.884
log(vocab_size) 10.8249
perplexity 53317.8
vocab_size 50257
perplexity / vocab_size 1.061
untrained perplexity sits near the vocabulary size


## Two documents in one sequence

The two documents are concatenated with an end-of-text token between them. Position `t` at that end-of-text token is a real token, and its target is the first token of the next document. That pair is an accident of packing: the model is being asked to continue one document with an unrelated document. Masking means setting that one target to `-100`.

On a fresh model every target is about equally surprising, so the mean barely moves. The comparison that shows the boundary is the one after each document has been learned on its own. The end-of-text token is the last token of each separate document, so the shift never teaches "end of document A predicts start of document B". That one target stays hard, and taking it out of the mean is the whole change.


In [6]:
doc_a = "The capital of France is Paris."
doc_b = "Taj Mahal stands in Agra."
eos = tokenizer.eos_token_id
ids_a = tokenizer.encode(doc_a, add_special_tokens=False) + [eos]
ids_b = tokenizer.encode(doc_b, add_special_tokens=False) + [eos]
packed = ids_a + ids_b
# Shifted index of the first token of document B. Logits at this index are the
# hidden state of document A's end-of-text token.
boundary = len(ids_a) - 1
packed_ids = torch.tensor(packed, dtype=torch.long, device=device).unsqueeze(0)
packed_mask = torch.ones_like(packed_ids)

print("packed tokens:")
print(" ".join(token_string(t) for t in packed))
print("boundary position", boundary)
print("context token", token_string(packed[boundary]))
print("target token ", token_string(packed[boundary + 1]))


def packed_losses(net):
    net.eval()
    with torch.no_grad():
        hidden_p = net.transformer(input_ids=packed_ids, attention_mask=packed_mask).last_hidden_state
        logits_p = net.lm_head(hidden_p)
        labels_p = packed_ids.clone()
        # No padding in this row. The only optional ignore is the boundary target.
        shift_logits = logits_p[:, :-1, :].reshape(-1, vocab_size).float()
        shift_labels = labels_p[:, 1:].reshape(-1)
        per_token = F.cross_entropy(shift_logits, shift_labels, reduction="none")
        loss_before = per_token.mean()
        scored = torch.ones_like(shift_labels, dtype=torch.bool)
        scored[boundary] = False
        loss_after = per_token[scored].mean()
    return loss_before, loss_after, per_token


before_raw, after_raw, _ = packed_losses(model)
print("untrained loss before boundary mask", round(before_raw.item(), 4))
print("untrained loss after boundary mask ", round(after_raw.item(), 4))

# Learn each document on its own, then measure the packed row.
pack_model = GPT2LMHeadModel(config).to(device)
pack_model.load_state_dict(model.state_dict())
sep_ids, sep_mask = pad_batch([ids_a, ids_b], pad_id)
opt_pack = torch.optim.AdamW(pack_model.parameters(), lr=3e-3)
pack_model.train()
for step in range(101):
    hidden_s = pack_model.transformer(input_ids=sep_ids, attention_mask=sep_mask).last_hidden_state
    logits_s = pack_model.lm_head(hidden_s)
    labels_s = sep_ids.clone()
    labels_s[sep_mask == 0] = -100
    step_loss, _, _, _, _ = shift_loss(logits_s, labels_s)
    if step % 25 == 0:
        print(f"separate-doc step {step:3d}  loss {step_loss.item():.4f}")
    if step == 100:
        break
    opt_pack.zero_grad(set_to_none=True)
    step_loss.backward()
    torch.nn.utils.clip_grad_norm_(pack_model.parameters(), 1.0)
    opt_pack.step()

loss_before, loss_after, per_token = packed_losses(pack_model)
n_before = int(per_token.numel())
n_after = n_before - 1
boundary_nll = float(per_token[boundary])
reconstructed = (float(loss_before) * n_before - boundary_nll) / n_after
print()
print("after learning each document separately")
print("loss before boundary mask", round(float(loss_before), 4))
print("loss after boundary mask ", round(float(loss_after), 4))
print("boundary target loss      ", round(boundary_nll, 4))
print("tokens before", n_before, "tokens after", n_after)
print("mean reconstructed from removing that one term", round(reconstructed, 4))
assert abs(reconstructed - float(loss_after)) < 1e-4
print()
print(f"{'pos':>4}  {'input token':<24}  {'target token':<24}  {'nll':>8}  role")
print("-" * 78)
for t in range(packed_ids.shape[1] - 1):
    role = "BOUNDARY" if t == boundary else "in-document"
    print(
        f"{t:4d}  {token_string(packed[t]):<24}  {token_string(packed[t + 1]):<24}  "
        f"{float(per_token[t]):8.3f}  {role}"
    )
print(
    "The masked target is the first token of document B, predicted from document A's "
    "end-of-text token. Separate training never built that pair: each document's "
    "end-of-text token was the last token in its own row, so the shift had nothing "
    "after it to score. That one term stays near log(vocab). Removing it, and only it, "
    "is the change from the before-mask mean to the after-mask mean; the reconstruction "
    "above is that subtraction. The first token of document B is also a bit above the "
    "other in-document terms, because GPT-2 positions are absolute: that token was "
    "trained at index 0 and sits later in the packed row."
)


packed tokens:
'The' ' capital' ' of' ' France' ' is' ' Paris' '.' '<|endoftext|>' 'T' 'aj' ' Mah' 'al' ' stands' ' in' ' Ag' 'ra' '.' '<|endoftext|>'
boundary position 7
context token '<|endoftext|>'
target token  'T'
untrained loss before boundary mask 10.8447
untrained loss after boundary mask  10.8517
separate-doc step   0  loss 10.8730
separate-doc step  25  loss 2.0826
separate-doc step  50  loss 0.0363
separate-doc step  75  loss 0.0085
separate-doc step 100  loss 0.0056

after learning each document separately
loss before boundary mask 0.7305
loss after boundary mask  0.1112
boundary target loss       10.6387
tokens before 17 tokens after 16
mean reconstructed from removing that one term 0.1112

 pos  input token               target token                   nll  role
------------------------------------------------------------------------------
   0  'The'                     ' capital'                   0.007  in-document
   1  ' capital'                ' of'                  

## Tied and untied heads

GPT-2's config defaults to `tie_word_embeddings=True`. After `post_init`, `lm_head.weight` and `transformer.wte.weight` are the same storage: `_tied_weights_keys` maps one onto the other, and `tie_weights` honors that only when the flag is true. `parameters()` skips the duplicate, so a tied model is smaller by exactly one `vocab_size * n_embd` matrix. Counting `named_parameters(remove_duplicate=False)` puts that matrix back and must match the untied model.


In [7]:
def parameter_count(net, remove_duplicate):
    return sum(p.numel() for _, p in net.named_parameters(remove_duplicate=remove_duplicate))

untied_config = GPT2Config(**{**config.to_dict(), "tie_word_embeddings": False})
untied_config.loss_type = "ForCausalLM"
untied_model = GPT2LMHeadModel(untied_config).to(device)

tied_params = parameter_count(model, remove_duplicate=True)
untied_params = parameter_count(untied_model, remove_duplicate=True)
tied_if_double_counted = parameter_count(model, remove_duplicate=False)
head_params = model.lm_head.weight.numel()

print("config.tie_word_embeddings", model.config.tie_word_embeddings)
print("_tied_weights_keys", model._tied_weights_keys)
print("tied, same storage", model.lm_head.weight.data_ptr() == model.transformer.wte.weight.data_ptr())
print("untied, same storage", untied_model.lm_head.weight.data_ptr() == untied_model.transformer.wte.weight.data_ptr())
print("tied parameter count  ", tied_params)
print("untied parameter count", untied_params)
print("extra untied matrix   ", untied_params - tied_params)
print("lm_head parameters    ", head_params)
print("tied count if the shared matrix is counted twice", tied_if_double_counted)
assert model.lm_head.weight.data_ptr() == model.transformer.wte.weight.data_ptr()
assert untied_model.lm_head.weight.data_ptr() != untied_model.transformer.wte.weight.data_ptr()
assert untied_params - tied_params == head_params == vocab_size * config.n_embd
assert tied_if_double_counted == untied_params
del untied_model


config.tie_word_embeddings True
_tied_weights_keys {'lm_head.weight': 'transformer.wte.weight'}
tied, same storage True
untied, same storage False
tied parameter count   3332928
untied parameter count 6549376
extra untied matrix    3216448
lm_head parameters     3216448
tied count if the shared matrix is counted twice 6549376


## Ordinary cross-entropy against a chunked one

The expensive tensor is the score matrix, shape `(tokens, vocab_size)`, plus the log-softmax that cross-entropy builds beside it. Chunking runs the same head on slices of the token axis, sums the token losses, and divides by the number of tokens that were not `-100`. The result matches the full call. Peak memory is the high-water mark of tensor storage alive during the call. On a GPU that is `max_memory_allocated`. On a CPU it is tracked with a dispatch mode that drops a storage when its tensor dies, so a chunk can be freed before the next one is built.


In [8]:
class PeakStorage(TorchDispatchMode):
    def __init__(self):
        self.live = {}
        self.peak = 0
        self._refs = []

    def _add(self, tensor):
        if not isinstance(tensor, torch.Tensor):
            return
        storage = tensor.untyped_storage()
        ptr = storage.data_ptr()
        nbytes = storage.nbytes()
        if ptr and nbytes and ptr not in self.live:
            self.live[ptr] = nbytes
            self.peak = max(self.peak, sum(self.live.values()))
            self._refs.append(weakref.ref(tensor, lambda _ref, p=ptr: self.live.pop(p, None)))

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        out = func(*args, **(kwargs or {}))
        stack = [out]
        while stack:
            item = stack.pop()
            if isinstance(item, torch.Tensor):
                self._add(item)
            elif isinstance(item, (list, tuple)):
                stack.extend(item)
            elif isinstance(item, dict):
                stack.extend(item.values())
        return out


def chunked_cross_entropy(hidden, weight, targets, chunk_size, ignore_index=-100):
    total = torch.zeros((), device=hidden.device, dtype=torch.float32)
    count = torch.zeros((), device=hidden.device, dtype=torch.long)
    for start in range(0, hidden.shape[0], chunk_size):
        logits = F.linear(hidden[start : start + chunk_size], weight).float()
        target_chunk = targets[start : start + chunk_size]
        total = total + F.cross_entropy(
            logits, target_chunk, ignore_index=ignore_index, reduction="sum"
        )
        count = count + (target_chunk != ignore_index).sum()
        del logits
    return total / count.clamp_min(1)


def ordinary_cross_entropy(hidden, weight, targets, ignore_index=-100):
    logits = F.linear(hidden, weight).float()
    return F.cross_entropy(logits, targets, ignore_index=ignore_index)


def measure_peak_bytes(fn):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        baseline = torch.cuda.memory_allocated()
        with torch.no_grad():
            loss = fn()
        torch.cuda.synchronize()
        return loss, torch.cuda.max_memory_allocated() - baseline
    tracker = PeakStorage()
    with tracker, torch.no_grad():
        loss = fn()
    return loss, tracker.peak


measure_tokens = 2048
chunk_size = 64
measure_hidden = torch.randn(measure_tokens, config.n_embd, device=device)
measure_weight = model.lm_head.weight.detach()
measure_targets = torch.randint(0, vocab_size, (measure_tokens,), device=device)
measure_targets[::17] = -100

ordinary_loss, ordinary_peak = measure_peak_bytes(
    lambda: ordinary_cross_entropy(measure_hidden, measure_weight, measure_targets)
)
chunked_loss, chunked_peak = measure_peak_bytes(
    lambda: chunked_cross_entropy(measure_hidden, measure_weight, measure_targets, chunk_size)
)
memory_ratio = ordinary_peak / chunked_peak
print("tokens", measure_tokens, "chunk", chunk_size, "vocab", vocab_size, "n_embd", config.n_embd)
print("ordinary loss", round(float(ordinary_loss), 6))
print("chunked loss ", round(float(chunked_loss), 6))
print("losses match", torch.allclose(ordinary_loss, chunked_loss, atol=1e-4, rtol=1e-4))
print(f"ordinary peak {ordinary_peak / 1024 ** 2:.2f} MB")
print(f"chunked peak  {chunked_peak / 1024 ** 2:.2f} MB")
print(f"ratio         {memory_ratio:.2f}x")
assert torch.allclose(ordinary_loss, chunked_loss, atol=1e-4, rtol=1e-4)
assert memory_ratio > 5


tokens 2048 chunk 64 vocab 50257 n_embd 64
ordinary loss 10.835638
chunked loss  10.835638
losses match True
ordinary peak 785.27 MB
chunked peak  25.06 MB
ratio         31.34x


## Part 1 numbers

These are the seven figures, then the memory pair and its ratio.


In [9]:
print("1  contributing tokens before padding mask ", contributing_before)
print("2  contributing tokens after padding mask  ", contributing_after)
print("3  packed loss before boundary mask        ", round(float(loss_before), 4))
print("4  packed loss after boundary mask         ", round(float(loss_after), 4))
print("5  untrained perplexity                    ", round(untrained_perplexity, 2))
print("6  tied parameter count                    ", tied_params)
print("7  untied parameter count                  ", untied_params)
print(f"   ordinary cross-entropy peak            {ordinary_peak / 1024 ** 2:.2f} MB")
print(f"   chunked cross-entropy peak             {chunked_peak / 1024 ** 2:.2f} MB")
print(f"   peak-memory ratio                      {memory_ratio:.2f}x")
print(f"   vocab size, for the perplexity check   {vocab_size}")


1  contributing tokens before padding mask  12
2  contributing tokens after padding mask   7
3  packed loss before boundary mask         0.7305
4  packed loss after boundary mask          0.1112
5  untrained perplexity                     53317.8
6  tied parameter count                     3332928
7  untied parameter count                   6549376
   ordinary cross-entropy peak            785.27 MB
   chunked cross-entropy peak             25.06 MB
   peak-memory ratio                      31.34x
   vocab size, for the perplexity check   50257


## Part 2: a second head, predicting token t+2

The hidden state at position `t` still goes to the original head, which scores token `t+1`. A second linear head sees the same vector and scores token `t+2`. The second head is a copy of the tied head's values, in a new storage. Sharing the parameter object would make one head, not two.

Both losses are means over their own valid targets. Padding is `-100` on the shifted mask, and the `t+2` head has no target at the last two positions, so it always has fewer contributing tokens. The training objective is the sum.


In [10]:
docs = [
    "The cat sat on the mat and watched the birds outside the window.",
    "Birds fly south in winter to find warmer weather and better food.",
    "A young girl opened a book and started to read beside the fireplace.",
    "The river runs through the valley after the rain filled the streams.",
    "Dogs bark at the moon when the night is still and unusually quiet.",
    "The baker made fresh bread before the sun came up over the village.",
    "Green leaves fall from the trees in late autumn and cover the ground.",
    "The boy kicked the ball across the empty field toward his friends.",
    "A small boat crossed the lake while the wind moved through the reeds.",
    "The teacher explained the problem slowly and answered every question.",
    "Clouds gathered above the hills before the first drops of rain.",
    "The farmer planted seeds in spring and waited for the harvest.",
]
encoded_docs = [tokenizer.encode(doc, add_special_tokens=False) for doc in docs]
train_ids, train_mask = pad_batch(encoded_docs, pad_id)
# Keep the toy sequences inside the context window.
assert train_ids.shape[1] <= config.n_positions

head_t2 = nn.Linear(config.n_embd, vocab_size, bias=False).to(device)
with torch.no_grad():
    head_t2.weight.copy_(model.lm_head.weight)
print("heads share storage", model.lm_head.weight.data_ptr() == head_t2.weight.data_ptr())
assert model.lm_head.weight.data_ptr() != head_t2.weight.data_ptr()

targets_t1 = train_ids[:, 1:].clone()
targets_t1[train_mask[:, 1:] == 0] = -100
targets_t2 = train_ids[:, 2:].clone()
targets_t2[train_mask[:, 2:] == 0] = -100
print("t+1 contributing tokens", int((targets_t1 != -100).sum()))
print("t+2 contributing tokens", int((targets_t2 != -100).sum()))

# One row, as strings, so the two shifts are visible.
print()
print(f"{'pos':>4}  {'input':<20}  {'t+1 target':<20}  {'t+2 target':<20}")
sample = 0
for t in range(min(8, train_ids.shape[1])):
    t1 = token_string(targets_t1[sample, t]) if t < targets_t1.shape[1] else "<none>"
    t2 = token_string(targets_t2[sample, t]) if t < targets_t2.shape[1] else "<none>"
    print(f"{t:4d}  {token_string(train_ids[sample, t]):<20}  {t1:<20}  {t2:<20}")

opt = torch.optim.AdamW(list(model.parameters()) + list(head_t2.parameters()), lr=1e-3)
model.train()
history = []
steps = 40
for step in range(steps + 1):
    hidden_b = model.transformer(input_ids=train_ids, attention_mask=train_mask).last_hidden_state
    logits_t1 = model.lm_head(hidden_b[:, :-1, :])
    logits_t2 = head_t2(hidden_b[:, :-2, :])
    loss_t1 = F.cross_entropy(
        logits_t1.reshape(-1, vocab_size).float(),
        targets_t1.reshape(-1),
        ignore_index=-100,
    )
    loss_t2 = F.cross_entropy(
        logits_t2.reshape(-1, vocab_size).float(),
        targets_t2.reshape(-1),
        ignore_index=-100,
    )
    total = loss_t1 + loss_t2
    history.append((step, float(loss_t1), float(loss_t2), float(total)))
    if step % 10 == 0 or step == steps:
        print(
            f"step {step:3d}  t+1={loss_t1.item():.4f}  "
            f"t+2={loss_t2.item():.4f}  sum={total.item():.4f}"
        )
    if step == steps:
        break
    opt.zero_grad(set_to_none=True)
    total.backward()
    torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(head_t2.parameters()), 1.0)
    opt.step()

start_t1, start_t2, start_sum = history[0][1], history[0][2], history[0][3]
end_t1, end_t2, end_sum = history[-1][1], history[-1][2], history[-1][3]

# Compare the heads on the positions that have both a t+1 and a t+2 target.
# The t+1 mean also includes the final real token of each row, which t+2 never scores.
model.eval()
with torch.no_grad():
    hidden_b = model.transformer(input_ids=train_ids, attention_mask=train_mask).last_hidden_state
    both = (targets_t1[:, :-1] != -100) & (targets_t2 != -100)
    only_t1 = (targets_t1 != -100) & ~torch.nn.functional.pad(both, (0, 1), value=False)
    logits_t1 = model.lm_head(hidden_b[:, :-1, :]).float()
    logits_t2 = head_t2(hidden_b[:, :-2, :]).float()
    nll_t1 = F.cross_entropy(
        logits_t1.reshape(-1, vocab_size), targets_t1.reshape(-1),
        ignore_index=-100, reduction="none",
    ).view_as(targets_t1)
    nll_t2 = F.cross_entropy(
        logits_t2.reshape(-1, vocab_size), targets_t2.reshape(-1),
        ignore_index=-100, reduction="none",
    ).view_as(targets_t2)
    matched_t1 = nll_t1[:, :-1][both].mean()
    matched_t2 = nll_t2[both].mean()
    tail_t1 = nll_t1[only_t1].mean()

print()
print("step 0   t+1", round(start_t1, 4), "t+2", round(start_t2, 4), "sum", round(start_sum, 4))
print("step", steps, " t+1", round(end_t1, 4), "t+2", round(end_t2, 4), "sum", round(end_sum, 4))
print("matched positions", int(both.sum()), "t+1", round(float(matched_t1), 4), "t+2", round(float(matched_t2), 4))
print("t+1-only tail positions", int(only_t1.sum()), "loss", round(float(tail_t1), 4))
print(
    f"Step 0 is log(vocab) for both heads ({math.log(vocab_size):.4f}): "
    f"t+1 {start_t1:.4f}, t+2 {start_t2:.4f}, sum {start_sum:.4f}. "
    "The second head is a copy of the untrained tied weights, so neither one starts with a next-token bias. "
    f"By step {steps} both have fallen by about five nats, to t+1 {end_t1:.4f} and t+2 {end_t2:.4f} "
    f"(sum {end_sum:.4f}). On the {int(both.sum())} positions that have both "
    f"a t+1 target and a t+2 target, the losses are {float(matched_t1):.4f} and {float(matched_t2):.4f}. "
    f"The positions only the t+1 head scores, the last real token of each row, sit at {float(tail_t1):.4f}. "
    "On this fixed batch both shifts are deterministic lookups from the same hidden state. "
    "Both heads began as the same random matrix, so the nearer token has no head start, and the curves stay together."
)


heads share storage False
t+1 contributing tokens 150
t+2 contributing tokens 138

 pos  input                 t+1 target            t+2 target          
   0  'The'                 ' cat'                ' sat'              
   1  ' cat'                ' sat'                ' on'               
   2  ' sat'                ' on'                 ' the'              
   3  ' on'                 ' the'                ' mat'              
   4  ' the'                ' mat'                ' and'              
   5  ' mat'                ' and'                ' watched'          
   6  ' and'                ' watched'            ' the'              
   7  ' watched'            ' the'                ' birds'            
step   0  t+1=10.8290  t+2=10.8237  sum=21.6528
step  10  t+1=9.5223  t+2=9.4734  sum=18.9957
step  20  t+1=8.3120  t+2=8.2202  sum=16.5322
step  30  t+1=6.9972  t+2=6.8753  sum=13.8725
step  40  t+1=5.7493  t+2=5.6092  sum=11.3586

step 0   t+1 10.829 t+2 10.8237 sum 21.6528
s

## Part 3: the shift that has to be read

`GPT2LMHeadModel.forward` says the labels are shifted inside the model, and that you may pass `labels = input_ids`. The implementation in this version of transformers does not slice the logits. `ForCausalLMLoss` pads a `-100` onto the right of the labels and then takes `labels[..., 1:]`, so the final logit is paired with "no target" and ignored. That is the same set of pairs as `logits[:, :-1]` against `tokens[:, 1:]`.

The mistake is to do both. If the labels you pass in are already the next-token column, the function shifts them once more. The mean loss of an untrained model barely changes, because every token is about equally wrong. The strings, and a row of logits that is deliberately confident about the true next token, make the off-by-one obvious.


In [11]:
print("GPT2LMHeadModel.forward says:")
for line in (GPT2LMHeadModel.forward.__doc__ or "").splitlines():
    if "shifted" in line or "labels = input_ids" in line or "-100" in line:
        print(" ", line.strip())

print()
print("ForCausalLMLoss does:")
source = inspect.getsource(ForCausalLMLoss)
for line in source.splitlines():
    if "shift" in line.lower() or "pad" in line or "ignore_index" in line or "view(-1" in line:
        print(" ", line.rstrip())

probe = "The capital of France is Paris."
probe_ids = tokenizer(probe, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    probe_hidden = model.transformer(input_ids=probe_ids).last_hidden_state
    probe_logits = model.lm_head(probe_hidden)
    hand_loss, _, _, _, _ = shift_loss(probe_logits, probe_ids)
    library_loss = model(input_ids=probe_ids, labels=probe_ids).loss
print()
print("hand loss, sliced once      ", round(float(hand_loss), 4))
print("library loss, labels=input   ", round(float(library_loss), 4))
assert torch.allclose(hand_loss, library_loss, atol=1e-5)

# A head that is sure of the true next token. Correct alignment must be near zero.
# A second shift scores the token after that, which this row assigns no score.
sure = torch.zeros(1, probe_ids.shape[1], vocab_size, device=device)
for t in range(probe_ids.shape[1] - 1):
    sure[0, t, probe_ids[0, t + 1]] = 50.0

correct = F.cross_entropy(sure[:, :-1, :].reshape(-1, vocab_size), probe_ids[:, 1:].reshape(-1))
# Already-shifted labels, length T, with -100 in the unused final slot.
# ForCausalLMLoss will pad and shift these again.
already = torch.full_like(probe_ids, -100)
already[:, :-1] = probe_ids[:, 1:]
once = ForCausalLMLoss(sure, probe_ids, vocab_size=vocab_size)
twice = ForCausalLMLoss(sure, already, vocab_size=vocab_size)
print("spiked logits, one shift    ", round(float(correct), 4))
print("spiked logits, library once ", round(float(once), 4))
print("spiked logits, shifted twice", round(float(twice), 4))
assert float(correct) < 0.1
assert torch.allclose(correct, once, atol=1e-4)
assert float(twice) > 10.0

library_targets = F.pad(already, (0, 1), value=-100)[..., 1:]
print()
print(f"{'pos':>4}  {'input':<22}  {'correct target':<22}  {'target after a second shift':<28}")
print("-" * 84)
for t in range(probe_ids.shape[1] - 1):
    print(
        f"{t:4d}  {token_string(probe_ids[0, t]):<22}  "
        f"{token_string(probe_ids[0, t + 1]):<22}  "
        f"{token_string(library_targets[0, t]):<28}"
    )
print(
    "One shift pairs each input with the next word. "
    "The second shift pairs it with the word after that, "
    "which is why the spiked row's loss jumps."
)


GPT2LMHeadModel.forward says:
  Labels for language modeling. Note that the labels **are shifted** inside the model, i.e. you can set
  `labels = input_ids` Indices are selected in `[-100, 0, ..., config.vocab_size]` All labels set to `-100`

ForCausalLMLoss does:
      ignore_index: int = -100,
      shift_labels: torch.Tensor | None = None,
      if shift_labels is None:
          # Shift so that tokens < n predict n
          labels = nn.functional.pad(labels, (0, 1), value=ignore_index)
          shift_labels = labels[..., 1:].contiguous()
      logits = logits.view(-1, vocab_size)
      shift_labels = shift_labels.view(-1)
      shift_labels = shift_labels.to(logits.device)
      loss = fixed_cross_entropy(logits, shift_labels, num_items_in_batch, ignore_index, **kwargs)

hand loss, sliced once       9.6348
library loss, labels=input    9.6348
spiked logits, one shift     0.0
spiked logits, library once  0.0
spiked logits, shifted twice 50.0

 pos  input                   correct 